In [13]:
print("hEy")

hEy


In [14]:
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

# =========================
# LOAD DATA
# =========================
matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

print("Initial shapes:", matches.shape, deliveries.shape)


Initial shapes: (1095, 20) (260920, 17)


In [15]:

# =========================
# PREPROCESSING
# =========================

# P1
matches = matches.dropna(subset=['winner'])

# P2, P3
matches['city'] = matches['city'].fillna('Unknown')
matches['method'] = matches['method'].fillna('Normal')

# P4
matches['date'] = pd.to_datetime(matches['date'])
matches['match_year'] = matches['date'].dt.year
matches['match_month'] = matches['date'].dt.month

# P5 - TEAM NORMALIZATION
team_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'
}

for col in ['team1','team2','toss_winner','winner']:
    matches[col] = matches[col].replace(team_map)

deliveries['batting_team'] = deliveries['batting_team'].replace(team_map)
deliveries['bowling_team'] = deliveries['bowling_team'].replace(team_map)

# P6
deliveries = deliveries[deliveries['inning'].isin([1,2])]

# P7, P8
deliveries['extras_type'] = deliveries['extras_type'].fillna('normal')
deliveries[['player_dismissed','dismissal_kind']] = deliveries[['player_dismissed','dismissal_kind']].fillna('none')


In [16]:

# =========================
# FEATURE ENGINEERING
# =========================

# merge
df = deliveries.merge(matches, left_on='match_id', right_on='id')

# TARGET (final score per innings)
final_score = df.groupby(['match_id','inning'])['total_runs'].sum().reset_index()
final_score.rename(columns={'total_runs':'final_score'}, inplace=True)

# FIRST 10 OVERS
df_10 = df[df['over'] < 10]

# AGG FEATURES
agg = df_10.groupby(['match_id','inning']).agg({
    'total_runs':'sum',
    'is_wicket':'sum',
    'batsman_runs':lambda x: ((x==4)|(x==6)).sum(),
    'extra_runs':'sum'
}).reset_index()

agg.columns = ['match_id','inning','runs_first10','wickets_first10','boundaries_first10','extras_first10']

# DOT BALLS
df_10['is_dot'] = (df_10['total_runs']==0).astype(int)
dots = df_10.groupby(['match_id','inning'])['is_dot'].sum().reset_index()
dots.rename(columns={'is_dot':'dots_first10'}, inplace=True)

# POWERPLAY
pp = df_10[df_10['over'] < 6].groupby(['match_id','inning']).agg({
    'total_runs':'sum',
    'is_wicket':'sum'
}).reset_index()
pp.columns = ['match_id','inning','powerplay_runs','powerplay_wickets']

# BATTERS USED
batters = df_10.groupby(['match_id','inning'])['batter'].nunique().reset_index()
batters.rename(columns={'batter':'batters_used_10'}, inplace=True)

# LEGAL BALLS
legal = df_10[~df_10['extras_type'].isin(['wides','noballs'])] \
    .groupby(['match_id','inning']).size().reset_index(name='legal_balls_10')

# MERGE ALL
features = final_score.merge(agg, on=['match_id','inning'])
features = features.merge(dots, on=['match_id','inning'])
features = features.merge(pp, on=['match_id','inning'])
features = features.merge(batters, on=['match_id','inning'])
features = features.merge(legal, on=['match_id','inning'])


C:\Users\shukl\AppData\Local\Temp\ipykernel_10664\1466221683.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_10['is_dot'] = (df_10['total_runs']==0).astype(int)


In [17]:

# =========================
# DERIVED FEATURES
# =========================

features['run_rate_10'] = features['runs_first10'] / (features['legal_balls_10']/6)
features['boundary_rate_10'] = features['boundaries_first10'] / (features['legal_balls_10']/6)
features['dot_rate_10'] = features['dots_first10'] / features['legal_balls_10']
features['pp_rr'] = features['powerplay_runs'] / 6


In [18]:

# =========================
# ADD MATCH CONTEXT
# =========================

match_info = matches[['id','venue','city','toss_winner','toss_decision',
                      'match_year','match_month','match_type']]

data = features.merge(match_info, left_on='match_id', right_on='id')


In [19]:

# =========================
# ENCODING
# =========================

encoders = {}
cat_cols = ['venue','city','toss_winner','toss_decision','match_type']

for col in cat_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le


In [20]:

# =========================
# FINAL DATASET
# =========================

X = data.drop(columns=['final_score','match_id','id'])
y = data['final_score']


In [21]:

# =========================
# TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [22]:

# =========================
# MODELS
# =========================

results = {}

def evaluate(name, model, Xtr, Xte):
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred)
    results[name] = {'MSE':mse, 'RMSE':rmse, 'R2':r2}
    return model

# Scaling for LR & SVR
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [23]:
# Models
lr = evaluate("Linear Regression", LinearRegression(), X_train_scaled, X_test_scaled)
print("Model Traind sucessfully\n")

Model Traind sucessfully



In [24]:
dt = evaluate("Decision Tree", DecisionTreeRegressor(max_depth=10, random_state=42), X_train, X_test)
print("Model Traind sucessfully\n")


Model Traind sucessfully



In [25]:
rf = evaluate("Random Forest", RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1), X_train, X_test)
print("Model Traind sucessfully\n")


Model Traind sucessfully



In [26]:
svr = evaluate("SVR", SVR(C=200, gamma=0.05, epsilon=0.1), X_train_scaled, X_test_scaled)
print("Model Traind sucessfully\n")



Model Traind sucessfully



In [27]:

# =========================
# STACKING
# =========================

# select top 2 by RMSE
sorted_models = sorted(results.items(), key=lambda x: x[1]['RMSE'])
best2 = [sorted_models[0][0], sorted_models[1][0]]

model_map = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "SVR": make_pipeline(StandardScaler(), SVR(C=200, gamma=0.05, epsilon=0.1))
}

estimators = [(name, model_map[name]) for name in best2]

stack = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression(),
    cv=5,
    passthrough=False
)

stack.fit(X_train, y_train)
pred = stack.predict(X_test)

mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, pred)

results["Stacking"] = {'MSE':mse, 'RMSE':rmse, 'R2':r2}



In [28]:
# =========================
# SAVE RESULTS
# =========================

output = {
    "results": results,
    "best2_models": best2,
    "n_train": len(X_train),
    "n_test": len(X_test),
    "features": list(X.columns)
}

with open("model_results.json", "w") as f:
    json.dump(output, f, indent=4)

print("✅ Done! Results saved to model_results.json")
pd.DataFrame(results)

✅ Done! Results saved to model_results.json


,Linear Regression,Decision Tree,Random Forest,SVR,Stacking
MSE,572.485886,750.892296,529.463562,657.743058,519.331279
RMSE,23.926677,27.402414,23.010075,25.646502,22.788841
R2,0.451875,0.281060,0.493067,0.370246,0.502768
